# Módulo 08 · Pipeline, línea de comandos y demostración web

`RAGPipeline` integra todas las piezas en una única interfaz.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
from fundamenta import PipelineConfig, RAGPipeline, SentenceTransformerEmbedder

pipeline = RAGPipeline.from_directory(
    DATA_DIR,
    embedder=SentenceTransformerEmbedder(),
    config=PipelineConfig(organization="NovaTech"),
)
print(f"{len(pipeline.chunks)} fragmentos indexados · modo {pipeline.mode}")

41 fragmentos indexados · modo hybrid


In [3]:
answer = pipeline.ask("¿Cuánto me pagan de dieta si viajo a Lisboa?", k=2)
print(answer.mode)
for source in answer.sources:
    print(f"[{source.number}] {source.label}")

extractive
[1] politica-de-gastos-y-viajes.md › Dietas
[2] politica-de-gastos-y-viajes.md › Transporte


Sin LLM configurado la respuesta es extractiva. Con `llm=ClaudeLLM()` u
`llm=OllamaLLM(...)` se genera una respuesta redactada con citas.

## Filtros

In [4]:
for result in pipeline.search("plazos", k=3, where={"source": "productos-y-precios.md"}):
    print(result.rank, result.document.metadata.get("section"))

1 Productos y precios de NovaTech > Soporte y SLA
2 Productos y precios de NovaTech > NovaDesk
3 Productos y precios de NovaTech > Política de cancelación


## Línea de comandos

```bash
fundamenta ask "¿Cuántos días de vacaciones tengo?" --embedder semantic --llm claude
fundamenta search "dietas en viajes" --k 3
fundamenta serve --embedder semantic --llm claude --port 8000
fundamenta eval --embedder semantic
```

## API web

El servidor de demostración expone la interfaz gráfica y una API JSON. Se arranca en
un puerto libre y se consulta desde el propio cuaderno:

In [5]:
import json
import threading
import urllib.request

from fundamenta.server import create_server

server = create_server(pipeline, port=0)
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_address[1]}"

with urllib.request.urlopen(url + "/api/health") as response:
    print(json.dumps(json.loads(response.read()), ensure_ascii=False, indent=2))

request = urllib.request.Request(
    url + "/api/ask",
    data=json.dumps({"question": "¿Qué hago si olvido la contraseña?"}).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request) as response:
    body = json.loads(response.read())
print("Modo:", body["mode"])
for source in body["sources"]:
    print(f"  [{source['number']}] {source['label']}: {source['text'][:60]}…")

server.shutdown()
server.server_close()

[fundamenta] 127.0.0.1 "GET /api/health HTTP/1.1" 200 -
{
  "status": "ok",
  "organization": "NovaTech",
  "chunks": 41,
  "retrieval": "hybrid",
  "generation": "extractive",
  "examples": []
}
[fundamenta] 127.0.0.1 "POST /api/ask HTTP/1.1" 200 -
Modo: extractive
  [1] faq-soporte-it.txt: ¿Qué hago si olvido mi contraseña?
Ve a login.novatech.examp…
  [2] faq-soporte-it.txt: ¿Cada cuánto tengo que cambiar mi contraseña?
Las contraseña…
  [3] faq-soporte-it.txt: ¿Qué hago si pierdo el portátil o me lo roban?
Avisa a IT de…
  [4] faq-soporte-it.txt: ¿Puedo instalar software por mi cuenta?
Solo desde el Centro…
